# Homeostasis and fault diagnostics

The figure generators below retain the published plotting code and formatting. The architecture-dependence panel now comes from paired live `MSNN` and `TemporalMCSNN` runs in the default reduced profile. Other quantitative panels likewise use genuine per-example arrays; expensive publication aggregates remain a separate replay path.

## Configuration and provenance

Saving is disabled by default. Reduced runs use a committed sub-megabyte fixture of genuine MNIST and Fashion-MNIST samples, plus moving-MNIST clips derived deterministically from those MNIST images. Published 20-seed aggregates are never expanded into synthetic seed observations.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import hashlib, json, os, shutil
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "experiments":
    ROOT = ROOT.parent
RUN_PROFILE = os.getenv("MNN_RUN_PROFILE", "reduced")  # reduced | publication | smoke
DEVICE = os.getenv("MNN_DEVICE", "auto")
WORKERS = os.getenv("MNN_WORKERS", "auto")
SAVE_FIGURES = os.getenv("MNN_SAVE_FIGURES", "0") == "1"
OUTPUT_DIR = Path(os.getenv("MNN_OUTPUT_DIR", str(ROOT / "experiments" / "generated_figures")))
OVERWRITE = os.getenv("MNN_OVERWRITE", "0") == "1"
RUN_EXTERNAL_DATA = os.getenv("MNN_RUN_EXTERNAL_DATA", "0") == "1"
WRITE_SAMPLE_CACHE = os.getenv("MNN_WRITE_HOMEOSTASIS_CACHE", "0") == "1"
ALLOW_DATA_DOWNLOADS = os.getenv("MNN_ALLOW_DATA_DOWNLOADS", "0") == "1"
assert RUN_PROFILE in {"reduced", "publication", "smoke"}

try:
    import torch
    if DEVICE == "auto":
        DEVICE_RESOLVED = "cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
    else:
        DEVICE_RESOLVED = DEVICE
except Exception:
    DEVICE_RESOLVED = "cpu"

MANIFEST = json.loads((ROOT / "experiments" / "figure_manifest.json").read_text())
SPEC = {row["filename"]: row for row in MANIFEST["figures"]}
FIGURE_REPORT = []

def _hash(value):
    if value is None:
        return None
    if isinstance(value, Path):
        return hashlib.sha256(value.read_bytes()).hexdigest() if value.exists() else None
    payload = json.dumps(value, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(payload).hexdigest()

def _guard_save(name, provenance):
    if provenance in {"placeholder", "reference-only", "external-gated"}:
        raise RuntimeError(f"Refusing to save non-claimable figure: {name}")
    if "manuscript" in str(OUTPUT_DIR).lower() and not OVERWRITE:
        raise RuntimeError("Writing into a manuscript directory requires MNN_OVERWRITE=1")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    target = OUTPUT_DIR / name
    if target.exists() and not OVERWRITE:
        raise FileExistsError(f"Refusing to overwrite {target}")
    return target

def finish(fig, name, data_source=None, reference=None, seeds=(), dpi=200, bbox_inches="tight", pad_inches=None,
           provenance_class=None, claim_status=None):
    spec = dict(SPEC[name])
    if provenance_class is not None:
        spec["provenance_class"] = provenance_class
    if claim_status is not None:
        spec["claim_status"] = claim_status
    display(fig)
    saved = None
    if SAVE_FIGURES:
        saved = _guard_save(name, spec["provenance_class"])
        save_kwargs={"dpi":dpi,"bbox_inches":bbox_inches,"facecolor":"white"}
        if pad_inches is not None: save_kwargs["pad_inches"]=pad_inches
        fig.savefig(saved, **save_kwargs)
    FIGURE_REPORT.append({
        **spec, "profile": RUN_PROFILE, "device": DEVICE_RESOLVED,
        "seeds": list(seeds), "data_sha256": _hash(data_source),
        "reference_sha256": _hash(reference), "saved_path": str(saved) if saved else None,
    })
    plt.close(fig)
    return None

def finish_asset(name, source):
    source = Path(source)
    display(Image(filename=str(source)))
    saved = None
    if SAVE_FIGURES:
        saved = _guard_save(name, SPEC[name]["provenance_class"])
        shutil.copyfile(source, saved)
    FIGURE_REPORT.append({**SPEC[name], "profile": RUN_PROFILE, "device": DEVICE_RESOLVED,
        "seeds": [], "data_sha256": None, "reference_sha256": _hash(source),
        "saved_path": str(saved) if saved else None})


## Live architecture-level homeostatic effect

The default reduced run executes paired homeostasis-off/on experiments for three tasks and seeds. Dense MNIST and Fashion-MNIST use `MSNN`; deterministic moving-MNIST clips use `TemporalMCSNN`. Each pair shares its initialization, sample selection, and loader order. The bar-chart body, labels, colours, error bars, and limits follow the publication generator, but its values and bootstrap intervals are computed from the live paired deltas. `publication` renders the recorded aggregate directly without pretending that the unavailable 20 seed rows survive.

In [ ]:
NOTEBOOK_NAME = "02_homeostasis.ipynb"
TEAL = "#3aa07a"; RED = "#c0392b"; BLUE = "#2f4b8f"; GREY = "#9aa6b2"; AMBER = "#e0a93b"
INK = "#2b2b2b"
homeostasis_reference = ROOT / "experiments" / "assets" / "references" / "fig_homeostasis_arch.png"
HOMEOSTASIS_FIXTURE = ROOT / "data" / "fixtures" / "homeostasis_reduced_datasets.npz"


def homeostasis_arch_budget(profile):
    if profile == "smoke":
        return {"seeds": [0], "dense_train": 80, "dense_test": 40,
                "dense_epochs": 1, "dense_steps": 6, "temporal_train": 80,
                "temporal_test": 40, "temporal_epochs": 1, "frames": 4,
                "batch": 20}
    return {"seeds": [0, 1, 2], "dense_train": 512, "dense_test": 256,
            "dense_epochs": 2, "dense_steps": 8, "temporal_train": 256,
            "temporal_test": 128, "temporal_epochs": 4, "frames": 6,
            "batch": 32}


def _load_homeostasis_fixture():
    if not HOMEOSTASIS_FIXTURE.exists():
        raise FileNotFoundError(
            f"Missing curated reduced dataset fixture: {HOMEOSTASIS_FIXTURE}")
    z = np.load(HOMEOSTASIS_FIXTURE, allow_pickle=False)
    metadata = json.loads(str(z["metadata_json"].item()))
    arrays = {key: z[key] for key in z.files if key != "metadata_json"}
    for task in ("mnist", "fashion"):
        for split in ("train", "test"):
            x, y = arrays[f"{task}_{split}_x"], arrays[f"{task}_{split}_y"]
            assert x.ndim == 4 and x.shape[1:] == (1, 28, 28)
            assert len(x) == len(y) and set(np.unique(y)) == set(range(10))
    return arrays, metadata


def _paired_subset(arrays, task, seed, train_n, test_n):
    rng_train = np.random.default_rng(seed)
    rng_test = np.random.default_rng(seed + 1000)
    train_idx = rng_train.choice(len(arrays[f"{task}_train_y"]), train_n, replace=False)
    test_idx = rng_test.choice(len(arrays[f"{task}_test_y"]), test_n, replace=False)
    train_x = torch.as_tensor(arrays[f"{task}_train_x"][train_idx], dtype=torch.float32).div_(255)
    train_y = torch.as_tensor(arrays[f"{task}_train_y"][train_idx], dtype=torch.long)
    test_x = torch.as_tensor(arrays[f"{task}_test_x"][test_idx], dtype=torch.float32).div_(255)
    test_y = torch.as_tensor(arrays[f"{task}_test_y"][test_idx], dtype=torch.long)
    return train_x, train_y, test_x, test_y


def _tensor_loaders(data, seed, batch):
    from torch.utils.data import DataLoader, TensorDataset
    train_x, train_y, test_x, test_y = data
    pin = DEVICE_RESOLVED == "cuda"
    train = DataLoader(TensorDataset(train_x, train_y), batch_size=batch,
                       shuffle=True, drop_last=True,
                       generator=torch.Generator().manual_seed(seed),
                       pin_memory=pin)
    test = DataLoader(TensorDataset(test_x, test_y), batch_size=batch,
                      shuffle=False, pin_memory=pin)
    return train, test


def _homeostasis_config(dev_params, homeo):
    from mnn_torch.training import make_config
    return make_config(
        dev_params, prob=0.0, homeo=homeo, disturb_conductance=False,
        pf_frozen_variability=True, homeostasis_threshold=5,
        homeostasis_r_th=0.8, homeostasis_n=1.0)


def _move(batch):
    return batch.to(DEVICE_RESOLVED, non_blocking=(DEVICE_RESOLVED == "cuda"))


def _train_dense_pair(task, data, seed, homeo, dev_params, budget):
    import torch.nn as nn
    from mnn_torch.models import MSNN
    torch.manual_seed(seed); np.random.seed(seed)
    train, test = _tensor_loaders(data, seed, budget["batch"])
    net = MSNN(28 * 28, 64, 10, num_steps=budget["dense_steps"], beta=.95,
               memristive_config=_homeostasis_config(dev_params, homeo)).to(DEVICE_RESOLVED)
    optimizer = torch.optim.Adam(net.parameters(), lr=8e-4)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(budget["dense_epochs"]):
        net.train()
        for images, labels in train:
            images, labels = _move(images), _move(labels)
            spikes, _ = net(images.reshape(images.shape[0], -1))
            loss = loss_fn(spikes.sum(0), labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    net.eval(); correct = total = 0
    with torch.no_grad():
        for images, labels in test:
            images, labels = _move(images), _move(labels)
            spikes, _ = net(images.reshape(images.shape[0], -1))
            correct += (spikes.sum(0).argmax(1) == labels).sum().item()
            total += labels.shape[0]
    return 100.0 * correct / total


def _translate_zero_fill(image, dx):
    shifted = torch.zeros_like(image)
    if dx >= 0:
        shifted[..., dx:] = image[..., :image.shape[-1] - dx]
    else:
        shifted[..., :dx] = image[..., -dx:]
    return shifted


def _moving_clips(images, labels, frames):
    base = np.linspace(-3, 2, frames).round().astype(int)
    clips = []
    for image, label in zip(images, labels):
        direction = 1 if int(label) % 2 == 0 else -1
        clips.append(torch.stack([_translate_zero_fill(image, direction * int(dx))
                                  for dx in base]))
    return torch.stack(clips)


def _temporal_loaders(data, seed, budget):
    from torch.utils.data import DataLoader, TensorDataset
    train_x, train_y, test_x, test_y = data
    train_x = _moving_clips(train_x, train_y, budget["frames"])
    test_x = _moving_clips(test_x, test_y, budget["frames"])
    pin = DEVICE_RESOLVED == "cuda"
    train = DataLoader(TensorDataset(train_x, train_y), batch_size=budget["batch"],
                       shuffle=True, drop_last=True,
                       generator=torch.Generator().manual_seed(seed), pin_memory=pin)
    test = DataLoader(TensorDataset(test_x, test_y), batch_size=budget["batch"],
                      shuffle=False, pin_memory=pin)
    return train, test


def _train_temporal_pair(data, seed, homeo, dev_params, budget):
    import torch.nn as nn
    from snntorch import surrogate
    from mnn_torch.models import TemporalMCSNN
    torch.manual_seed(seed); np.random.seed(seed)
    train, test = _temporal_loaders(data, seed, budget)
    config = {**_homeostasis_config(dev_params, homeo), "conv_ideal": True}
    net = TemporalMCSNN(
        beta=.95, spike_grad=surrogate.fast_sigmoid(slope=25),
        num_steps=budget["frames"], batch_size=budget["batch"], num_kernels=3,
        num_conv1=4, num_conv2=8, max_pooling=2, num_outputs=10,
        memristive_config=config).to(DEVICE_RESOLVED)
    optimizer = torch.optim.Adam(net.parameters(), lr=8e-4)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(budget["temporal_epochs"]):
        net.train()
        for clips, labels in train:
            clips, labels = _move(clips), _move(labels)
            spikes, _, _ = net(clips.permute(1, 0, 2, 3, 4))
            loss = loss_fn(spikes.sum(0), labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    net.eval(); correct = total = 0
    with torch.no_grad():
        for clips, labels in test:
            clips, labels = _move(clips), _move(labels)
            spikes, _, _ = net(clips.permute(1, 0, 2, 3, 4))
            correct += (spikes.sum(0).argmax(1) == labels).sum().item()
            total += labels.shape[0]
    return 100.0 * correct / total


def _bootstrap_mean_ci(values, n=5000, seed=0):
    values = np.asarray(values, dtype=float)
    if len(values) == 1:
        return float(values[0]), float(values[0])
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), (n, len(values)))].mean(1)
    return tuple(np.percentile(means, [2.5, 97.5]).tolist())


def run_live_homeostasis_architecture(profile):
    import time
    from mnn_torch.training import precompute_device_params
    arrays, fixture_metadata = _load_homeostasis_fixture()
    budget = homeostasis_arch_budget(profile)
    dev_params = precompute_device_params(ROOT / "data")
    task_specs = [
        ("MSNN MNIST", "mnist", "dense"),
        ("MSNN Fashion-MNIST", "fashion", "dense"),
        ("MCSNN moving-MNIST", "mnist", "temporal"),
    ]
    started = time.time(); tasks = {}
    for title, fixture_task, architecture in task_specs:
        rows = []
        for seed in budget["seeds"]:
            if architecture == "dense":
                data = _paired_subset(arrays, fixture_task, seed,
                                      budget["dense_train"], budget["dense_test"])
                off = _train_dense_pair(fixture_task, data, seed, False, dev_params, budget)
                on = _train_dense_pair(fixture_task, data, seed, True, dev_params, budget)
            else:
                data = _paired_subset(arrays, fixture_task, seed,
                                      budget["temporal_train"], budget["temporal_test"])
                off = _train_temporal_pair(data, seed, False, dev_params, budget)
                on = _train_temporal_pair(data, seed, True, dev_params, budget)
            rows.append({"seed": seed, "without_homeostasis": off,
                         "with_homeostasis": on, "benefit_pp": on - off})
        deltas = [row["benefit_pp"] for row in rows]
        lo, hi = _bootstrap_mean_ci(deltas)
        tasks[title] = {"rows": rows, "benefit": float(np.mean(deltas)),
                        "ci": [lo, hi]}
    return {"profile": profile, "device": DEVICE_RESOLVED, "budget": budget,
            "fixture_metadata": fixture_metadata, "tasks": tasks,
            "wall_seconds": time.time() - started}


if RUN_PROFILE == "publication":
    summary_path = ROOT / "data" / "results" / "publication_summary.json"
    summary = json.loads(summary_path.read_text())["homeostasis_architecture"]
    ARCHITECTURE_RESULTS = {
        "profile": "publication", "tasks": {
            key: {"benefit": value["benefit"], "ci": value["ci"], "rows": []}
            for key, value in summary.items() if key != "source"}}
    architecture_source = summary_path
    architecture_seeds = []
    architecture_provenance = "published-aggregate"
    architecture_claim = "claimable-from-published-aggregate"
else:
    ARCHITECTURE_RESULTS = run_live_homeostasis_architecture(RUN_PROFILE)
    architecture_source = ARCHITECTURE_RESULTS
    architecture_seeds = ARCHITECTURE_RESULTS["budget"]["seeds"]
    architecture_provenance = "live-reduced"
    architecture_claim = "reduced-validation"


# Appendix-faithful plotting body; only the data source is now live in reduced/smoke.
task_order = ["MSNN MNIST", "MSNN Fashion-MNIST", "MCSNN moving-MNIST"]
labels = ["MSNN\n(dense)\nMNIST", "MSNN\n(dense)\nFashion",
          "TemporalMCSNN\n(temporal-conv)\nmoving-MNIST"]
delta = [ARCHITECTURE_RESULTS["tasks"][key]["benefit"] for key in task_order]
lo = [ARCHITECTURE_RESULTS["tasks"][key]["ci"][0] for key in task_order]
hi = [ARCHITECTURE_RESULTS["tasks"][key]["ci"][1] for key in task_order]
cols = [TEAL if value > 0 else GREY for value in delta]
fig, ax = plt.subplots(figsize=(5.6, 4.0))
x = np.arange(len(labels))
err = np.array([[value - lower for value, lower in zip(delta, lo)],
                [upper - value for value, upper in zip(delta, hi)]])
ax.bar(x, delta, color=cols, edgecolor=INK, linewidth=0.6, yerr=err,
       capsize=4, error_kw=dict(ecolor=INK, lw=1))
ax.axhline(0, color=INK, lw=1)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel("homeostatic benefit (pp)\n[homeo $-$ no-homeo]", fontsize=9)
ax.set_title("Architecture dependence of the homeostatic benefit", fontsize=10)
fig.tight_layout()
finish(fig, "fig_homeostasis_arch.png", architecture_source, homeostasis_reference,
       seeds=architecture_seeds, dpi=200, provenance_class=architecture_provenance,
       claim_status=architecture_claim)
{"profile": ARCHITECTURE_RESULTS["profile"],
 "device": ARCHITECTURE_RESULTS.get("device", "archive"),
 "wall_seconds": ARCHITECTURE_RESULTS.get("wall_seconds"),
 "tasks": {key: {"benefit": value["benefit"], "ci": value["ci"],
                  "paired_seed_deltas": [row["benefit_pp"] for row in value["rows"]]}
           for key, value in ARCHITECTURE_RESULTS["tasks"].items()}}


## Asset-versus-liability mechanism

This is the authored two-panel generator used for the publication figure, retained with its original palette, geometry, equations, and annotations.

In [ ]:
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.4))
for ax in (axL, axR):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")

# Shared device-transient glyph (rise then slow decay).
def transient(ax, x0, y0, w, h, colour, decay=True):
    t = np.linspace(0, 1, 200)
    rise = 1 - np.exp(-t / 0.08)
    if decay:
        y = rise * np.exp(-t / 0.5)
        y = y / y.max()
    else:
        y = rise
    ax.plot(x0 + t * w, y0 + y * h, color=colour, lw=2.0)

# (a) LIABILITY
axL.set_title(r"(a) Static use $\rightarrow$ the transient is a LIABILITY", fontsize=11, color=INK, loc="left")
axL.add_patch(FancyBboxPatch((0.03, 0.60), 0.20, 0.22, boxstyle="round,pad=0.01", fc="#eef1f4", ec=GREY, lw=1.2))
axL.text(0.13, 0.71, "static\ninput\n(same every step)", ha="center", va="center", fontsize=8.5, color=INK)
axL.add_patch(FancyBboxPatch((0.33, 0.58), 0.22, 0.26, boxstyle="round,pad=0.01", fc="#fdecea", ec=RED, lw=1.4))
axL.text(0.44, 0.79, "memristive synapse", ha="center", fontsize=8.5, color=INK)
transient(axL, 0.35, 0.60, 0.18, 0.12, RED)
axL.text(0.44, 0.545, "stuck / hyperactive device", ha="center", fontsize=7.5, color=RED)
axL.add_patch(FancyBboxPatch((0.66, 0.58), 0.22, 0.26, boxstyle="round,pad=0.01", fc="#fdecea", ec=RED, lw=1.4))
axL.text(0.77, 0.79, "postsynaptic neuron", ha="center", fontsize=8.5, color=INK)
sp = np.linspace(0.68, 0.86, 22)
for x in sp:
    axL.plot([x, x], [0.61, 0.71], color=RED, lw=1.0)
axL.text(0.77, 0.545, "spike-rate runaway", ha="center", fontsize=7.5, color=RED)
for x0, x1 in [(0.235, 0.33), (0.55, 0.66)]:
    axL.add_patch(FancyArrowPatch((x0, 0.71), (x1, 0.71), arrowstyle="-|>", mutation_scale=14, color=INK, lw=1.2))
axL.add_patch(FancyBboxPatch((0.30, 0.16), 0.40, 0.20, boxstyle="round,pad=0.01", fc="#eaf5ef", ec=TEAL, lw=1.6))
axL.text(0.50, 0.29, "homeostatic regulariser", ha="center", fontsize=9, color=TEAL)
axL.text(0.50, 0.215, r"$G_{\rm steady}=G_{\max}/[1+(r_{\rm in}/r_{\rm th})^{n}]$", ha="center", fontsize=8.5, color=INK)
axL.add_patch(FancyArrowPatch((0.60, 0.365), (0.77, 0.575), arrowstyle="-|>", mutation_scale=13, color=TEAL, lw=1.4))
axL.text(0.055, 0.06, "helps dense (+2.6/+4.4 pp) . hurts temporal-conv (-11.3 pp)", fontsize=8, color=INK)

# (b) ASSET
axR.set_title(r"(b) Temporal use $\rightarrow$ the same relaxation is an ASSET", fontsize=11, color=INK, loc="left")
axR.add_patch(FancyBboxPatch((0.03, 0.60), 0.20, 0.22, boxstyle="round,pad=0.01", fc="#eaf0fb", ec=BLUE, lw=1.2))
axR.text(0.13, 0.71, "streamed\ninput\n(unfolds in time)", ha="center", va="center", fontsize=8.5, color=INK)
axR.add_patch(FancyBboxPatch((0.33, 0.58), 0.30, 0.26, boxstyle="round,pad=0.01", fc="#eaf5ef", ec=TEAL, lw=1.6))
axR.text(0.48, 0.79, "LIF membrane leak", ha="center", fontsize=8.5, color=INK)
transient(axR, 0.35, 0.60, 0.26, 0.12, TEAL)
axR.text(0.48, 0.545, r"$\beta=\exp(-\Delta t/\tau_{\rm leak})$", ha="center", fontsize=8.5, color=TEAL)
axR.add_patch(FancyArrowPatch((0.235, 0.71), (0.33, 0.71), arrowstyle="-|>", mutation_scale=14, color=INK, lw=1.2))
axR.text(0.83, 0.80, "hold across delay $D$", ha="center", fontsize=8, color=INK)
g = np.linspace(0, 1, 100)
axR.plot(0.70 + g * 0.26, 0.63 + 0.14 * np.exp(-g / 0.7), color=TEAL, lw=2.2)
axR.plot(0.70 + g * 0.26, 0.63 + 0.14 * np.exp(-g / 0.12), color=GREY, lw=1.6, ls="--")
axR.text(0.98, 0.70, r"long $\tau$", fontsize=7.5, color=TEAL, ha="right")
axR.text(0.98, 0.635, r"short $\tau$", fontsize=7.5, color=GREY, ha="right")
axR.add_patch(FancyArrowPatch((0.63, 0.71), (0.70, 0.71), arrowstyle="-|>", mutation_scale=14, color=INK, lw=1.2))
axR.add_patch(FancyBboxPatch((0.30, 0.16), 0.40, 0.20, boxstyle="round,pad=0.01", fc="#eaf5ef", ec=TEAL, lw=1.6))
axR.text(0.50, 0.29, "retention as an asset", ha="center", fontsize=9, color=TEAL)
axR.text(0.50, 0.215, r"$D_{\max}\approx s_D\,\tau_{\rm leak}$", ha="center", fontsize=9, color=INK)
axR.text(0.055, 0.06, r"longer device retention $\rightarrow$ longer bridgeable delay", fontsize=8, color=INK)

fig.tight_layout(rect=(0, 0, 1, 0.97))
fig.lines.append(plt.Line2D([0.5, 0.5], [0.10, 0.90], transform=fig.transFigure, color=GREY, lw=1.0, ls=(0, (4, 4))))
fig.text(0.5, 0.50, "one measured\n" + r"$\tau_{\rm leak}$", ha="center", va="center", fontsize=9.5, color=INK, bbox=dict(boxstyle="round,pad=0.35", fc="#fff6e6", ec=AMBER, lw=1.8))
fig.patches.append(FancyArrowPatch((0.47, 0.50), (0.40, 0.50), transform=fig.transFigure, arrowstyle="-|>", mutation_scale=12, color=AMBER, lw=1.3))
fig.patches.append(FancyArrowPatch((0.53, 0.50), (0.60, 0.50), transform=fig.transFigure, arrowstyle="-|>", mutation_scale=12, color=AMBER, lw=1.3))
finish(fig, "fig_liability_asset.png", {"generator": "gen_liability_asset", "version": 1}, ROOT / "experiments" / "assets" / "references" / "fig_liability_asset.png", dpi=200, bbox_inches=None)


## Fault-condition confusion matrices

These matrices are computed from per-example targets and predictions returned by genuine `run_condition` training jobs. Publication mode replays a committed sample archive; reduced mode runs bounded MNIST jobs; smoke mode trains on a deterministic offline fixture.

In [ ]:
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from snntorch import surrogate
from torch.utils.data import DataLoader, TensorDataset
from mnn_torch.data import mnist_loaders
from mnn_torch.models import MCSNN
from mnn_torch.training import precompute_device_params, run_condition

SAMPLE_CACHE = ROOT / "data" / "results" / "homeostasis_sample_archive.npz"

def offline_fixture(seed=0, n_train=80, n_test=40):
    rng = np.random.default_rng(seed)
    def make(n):
        y = np.arange(n, dtype=np.int64) % 10
        x = rng.normal(0.0, 0.05, (n, 1, 28, 28)).astype(np.float32)
        for i, label in enumerate(y):
            col = 2 + 2 * int(label)
            x[i, 0, 4:24, col:col + 3] += 1.0
            x[i, 0, 4 + int(label):7 + int(label), 3:25] += 0.5
        return x, y
    train_x, train_y = make(n_train); test_x, test_y = make(n_test)
    return {"train_x": train_x, "train_y": train_y, "test_x": test_x, "test_y": test_y}

def conv2_sample_rates(images, labels, seed, steps):
    torch.manual_seed(seed); np.random.seed(seed)
    batch = min(16, len(images))
    loader = DataLoader(TensorDataset(torch.as_tensor(images), torch.as_tensor(labels)), batch_size=batch, shuffle=False)
    net = MCSNN(beta=.95, spike_grad=surrogate.fast_sigmoid(slope=25), num_steps=steps, batch_size=batch,
        num_kernels=3, num_conv1=4, num_conv2=8, max_pooling=2, num_outputs=10,
        memristive_config={"ideal": True, "homeostasis_dropout": True, "homeostasis_threshold": 2}).to(DEVICE_RESOLVED)
    # Deterministic gain calibration keeps the compact one-step run above the LIF threshold.
    with torch.no_grad(): net.conv1.weight.mul_(2.0); net.conv2.weight.mul_(2.0)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3); loss_fn = nn.CrossEntropyLoss()
    bx, by = next(iter(loader)); bx = bx.to(DEVICE_RESOLVED); by = by.to(DEVICE_RESOLVED)
    net.train(); spk_out, _, _ = net(bx); loss = loss_fn(spk_out.sum(0), by)
    opt.zero_grad(); loss.backward(); opt.step(); net.eval()
    with torch.no_grad(): _, _, spk_int = net(bx)
    sample = min(5, spk_int.shape[1] - 1)
    rates = spk_int[:, sample].reshape(spk_int.shape[0], spk_int.shape[2], -1).sum(-1)
    return rates.cpu().numpy(), float(loss.detach().cpu()), sample

def compute_sample_archive(profile):
    seed = 0
    if profile == "smoke":
        fixture = offline_fixture(seed); train_n, test_n, epochs, steps, batch = 80, 40, 1, 8, 20
        feature_x, feature_y = fixture["test_x"], fixture["test_y"]
    else:
        fixture = None; train_n, test_n, epochs, steps, batch = 256, 128, 1, 8, 32
        _, test_loader = mnist_loaders(ROOT / "data", seed=seed, batch_size=test_n, train_subset=train_n, test_subset=test_n)
        feature_x, feature_y = next(iter(test_loader)); feature_x, feature_y = feature_x.numpy(), feature_y.numpy()
    dev_params = precompute_device_params(ROOT / "data")
    common = dict(seed=seed, prob=.8, disturb_mode="device_fixed", stuck_polarity="on", polarity="on",
        epochs=epochs, num_steps=steps, train_subset=train_n, test_subset=test_n, batch_size=batch, lr=5e-4, data=str(ROOT / "data"),
        device=DEVICE_RESOLVED, threads=1, dev_params=dev_params, fixture=fixture, return_predictions=True)
    arms = {}
    for homeo in (False, True):
        result = run_condition({**common, "homeo": homeo})
        arms["homeo" if homeo else "no_homeo"] = result
    rates, feature_loss, sample = conv2_sample_rates(feature_x, feature_y, seed, steps)
    metadata = {"schema_version": 1, "profile": profile, "seed": seed,
        "dataset": "offline deterministic fixture" if fixture is not None else "MNIST test split", "train_subset": train_n,
        "test_subset": test_n, "epochs": epochs, "num_steps": steps, "batch_size": batch, "fault_probability": .8,
        "disturb_mode": "device_fixed", "stuck_polarity": "on", "feature_sample_index": sample,
        "feature_model": "MCSNN(4,8 conv channels; one genuine optimisation step)", "conv_weight_scale": 2.0,
        "feature_loss": feature_loss, "device": DEVICE_RESOLVED}
    return {"targets_no_homeo": arms["no_homeo"]["targets"], "predictions_no_homeo": arms["no_homeo"]["predictions"],
        "targets_homeo": arms["homeo"]["targets"], "predictions_homeo": arms["homeo"]["predictions"],
        "conv2_spike_rates": rates, "metadata": metadata}

if RUN_PROFILE == "publication" and SAMPLE_CACHE.exists():
    z = np.load(SAMPLE_CACHE, allow_pickle=False)
    samples = {k: z[k] for k in z.files if k != "metadata_json"}; samples["metadata"] = json.loads(str(z["metadata_json"].item()))
else:
    samples = compute_sample_archive(RUN_PROFILE)
sample_hasher = hashlib.sha256()
for key in ("targets_no_homeo", "predictions_no_homeo", "targets_homeo", "predictions_homeo", "conv2_spike_rates"):
    sample_hasher.update(np.ascontiguousarray(samples[key]).tobytes())
sample_hasher.update(json.dumps(samples["metadata"], sort_keys=True).encode())
sample_provenance = {"archive_sha256": sample_hasher.hexdigest(), "metadata": samples["metadata"]}
if WRITE_SAMPLE_CACHE:
    SAMPLE_CACHE.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(SAMPLE_CACHE, **{k:v for k,v in samples.items() if k != "metadata"}, metadata_json=json.dumps(samples["metadata"], sort_keys=True))
sample_data_source = SAMPLE_CACHE if RUN_PROFILE in {"publication", "reduced"} and SAMPLE_CACHE.exists() else sample_provenance

fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, key, panel in zip(axes, ("no_homeo", "homeo"), ("(a)", "(b)")):
    cm = confusion_matrix(samples[f"targets_{key}"], samples[f"predictions_{key}"], labels=np.arange(10))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.arange(10)).plot(ax=ax, cmap="viridis", colorbar=True)
    ax.set_title(panel, loc="left", fontweight="bold")
fig.tight_layout()
finish(fig, "fig_confusion-faults.png", sample_data_source, ROOT / "experiments" / "assets" / "references" / "fig_confusion-faults.png", seeds=[samples["metadata"]["seed"]], dpi=200, bbox_inches=None)


## Second-convolution feature maps

The plotting body is the original second-convolution trace code. Its sample array is produced above by an actual `MCSNN` forward pass after a genuine optimisation step and stored with complete run metadata.

In [ ]:
def plot_second_conv_feature_maps(spike_rate):
    fig = plt.figure(figsize=(10, 5))
    for ch in range(spike_rate.shape[1]):
        plt.plot(spike_rate[:, ch], label=f'Channel {ch}')
    plt.title("Feature Maps at Second Convolutional Layer")
    plt.xlabel("Time Step")
    plt.ylabel("Spike Count")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    return fig

feature_fig = plot_second_conv_feature_maps(samples["conv2_spike_rates"])
finish(feature_fig, "homeostatic_feature_maps.png", sample_data_source, ROOT / "experiments" / "assets" / "references" / "homeostatic_feature_maps.png", seeds=[samples["metadata"]["seed"]], dpi=200, bbox_inches=None)


## Reproduction report

In [ ]:
assert len(FIGURE_REPORT) == len([x for x in MANIFEST["figures"] if x["notebook"] == Path(globals().get("NOTEBOOK_NAME", "")).name])
FIGURE_REPORT